# День 4 — baseline без дообучения

DistilBERT только считает CLS-эмбеддинги. Классификатор — логистическая регрессия на этих векторах.

Метки датасета: −1 negative, 0 neutral, 1 positive. Пустые строки пропускаем.

## Задача 1. Токенизация

In [1]:
import sys
from pathlib import Path

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from transformers import AutoModel, AutoTokenizer

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.baseline import LABEL_NAMES, get_cls_embeddings, load_twitter
from src.embeddings import load_model
from src.tokenization import MODEL_NAME, load_tokenizer, tokenize_texts

tokenizer = load_tokenizer(MODEL_NAME)
print(AutoTokenizer.__name__, tokenizer.name_or_path)

examples = [
    "This movie was absolutely amazing!",
    "Terrible movie, waste of time.",
    "Pretty good, I liked it.",
]
example_tokens = tokenize_texts(examples, tokenizer)
print(example_tokens["input_ids"])
print(f"Shape: {example_tokens['input_ids'].shape}")

AutoTokenizer distilbert-base-uncased
tensor([[ 101, 2023, 3185, 2001, 7078, 6429,  999,  102,    0],
        [ 101, 6659, 3185, 1010, 5949, 1997, 2051, 1012,  102],
        [ 101, 3492, 2204, 1010, 1045, 4669, 2009, 1012,  102]])
Shape: torch.Size([3, 9])


## Задача 2. CLS-эмбеддинги

In [2]:
model = load_model(MODEL_NAME)
print(AutoModel.__name__, model.name_or_path)
print(f"training: {model.training}")

example_embeddings = get_cls_embeddings(examples, tokenizer, model)
print(f"Embeddings shape: {example_embeddings.shape}")
print("Ожидается: (3, 768) для DistilBERT")

W1023 18:07:09.176000 44366 .venv/lib/python3.12/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


AutoModel distilbert-base-uncased
training: False


3/3


Embeddings shape: (3, 768)
Ожидается: (3, 768) для DistilBERT


## Задача 3. Logistic Regression

Эмбеддинги всего датасета кэшируются в `outputs/baseline/`. Повторный запуск читает кэш, если число строк совпало.

In [3]:
texts, labels = load_twitter(ROOT / "Twitter_Data.csv")
labels = np.asarray(labels)
print(f"Текстов: {len(texts)}")
print({name: int((labels == code).sum()) for code, name in LABEL_NAMES.items()})

cache_dir = ROOT / "outputs" / "baseline"
cache_dir.mkdir(parents=True, exist_ok=True)
embeddings_path = cache_dir / "cls_embeddings.npy"
labels_path = cache_dir / "labels.npy"

if embeddings_path.exists() and labels_path.exists():
    embeddings = np.load(embeddings_path)
    cached_labels = np.load(labels_path)
    cache_ok = len(embeddings) == len(texts) and np.array_equal(cached_labels, labels)
else:
    cache_ok = False

if cache_ok:
    print(f"Кэш: {embeddings.shape}")
else:
    embeddings = get_cls_embeddings(texts, tokenizer, model)
    np.save(embeddings_path, embeddings)
    np.save(labels_path, labels)
    print(f"Сохранено: {embeddings.shape}")

X_train, X_test, y_train, y_test = train_test_split(
    embeddings,
    labels,
    test_size=0.2,
    stratify=labels,
    random_state=42,
)
print(f"Train: {X_train.shape}, test: {X_test.shape}")

classifier = LogisticRegression(max_iter=1000, n_jobs=-1)
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)

report = classification_report(
    y_test,
    y_pred,
    labels=[-1, 0, 1],
    target_names=[LABEL_NAMES[-1], LABEL_NAMES[0], LABEL_NAMES[1]],
)
macro_f1 = f1_score(y_test, y_pred, average="macro")
print(report)
print(f"macro F1: {macro_f1:.4f}")

results_path = ROOT / "baseline_results.txt"
results_path.write_text(f"{report}\nmacro F1: {macro_f1:.4f}\n", encoding="utf-8")
print(results_path)

Текстов: 162968
{'negative': 35509, 'neutral': 55210, 'positive': 72249}


2048/162968


4096/162968


6144/162968


8192/162968


...

161792/162968


162968/162968


Сохранено: (162968, 768)
Train: (130374, 768), test: (32594, 768)


/Users/vladimir.fedorov/fourth_task_transformers/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


              precision    recall  f1-score   support

    negative       0.54      0.40      0.46      7102
     neutral       0.67      0.70      0.69     11042
    positive       0.67      0.73      0.70     14450

    accuracy                           0.65     32594
   macro avg       0.63      0.61      0.61     32594
weighted avg       0.64      0.65      0.64     32594

macro F1: 0.6147
/Users/vladimir.fedorov/fourth_task_transformers/baseline_results.txt
